## clean nav_history.csv ##

In [84]:
import pandas as pd
df = pd.read_csv("../data/raw/02_nav_history.csv")
df.head()


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [85]:
print("original shape:", df.shape)

original shape: (46000, 3)


In [86]:
# converting date column to datetime
df["date"] = pd.to_datetime(df['date'])

In [87]:
#sorting by amfi_code and date
df = df.sort_values(['amfi_code','date'])

In [88]:
#check for duplicates
before  = len(df)
print(before)

df = df.drop_duplicates()
print("duplicates removed", before - len(df))



46000
duplicates removed 0


In [89]:
# ensure nav > 0
invalid_nav = df[df['nav']<=0]
print(invalid_nav)

df = df[df['nav']>0]




Empty DataFrame
Columns: [amfi_code, date, nav]
Index: []


In [90]:
missing_before = df['nav'].isnull().sum()
missing_before

np.int64(0)

In [91]:
print("final shape", df.shape)

final shape (46000, 3)


In [92]:
print(cleaned)

None


In [93]:
df.info()
print(df.dtypes)

<class 'pandas.core.frame.DataFrame'>
Index: 46000 entries, 5750 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.4 MB
amfi_code             int64
date         datetime64[ns]
nav                 float64
dtype: object


In [94]:
df.to_csv("../data/processed/02_nav_history_clean.csv", index=False)
print("Saved cleaned nav_history to data/processed/")

Saved cleaned nav_history to data/processed/


In [95]:
clean_df = pd.read_csv(
    "../data/processed/02_nav_history_clean.csv",
    parse_dates=["date"]
)

clean_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   amfi_code  46000 non-null  int64         
 1   date       46000 non-null  datetime64[ns]
 2   nav        46000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 1.1 MB


## clean investor transaction ##

In [96]:
df2 = pd.read_csv("../data/raw/08_investor_transactions.csv")
df2.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [97]:
print("original shape:", df2.shape)

original shape: (32778, 13)


In [99]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   investor_id         32778 non-null  object 
 1   transaction_date    32778 non-null  object 
 2   amfi_code           32778 non-null  int64  
 3   transaction_type    32778 non-null  object 
 4   amount_inr          32778 non-null  int64  
 5   state               32778 non-null  object 
 6   city                32778 non-null  object 
 7   city_tier           32778 non-null  object 
 8   age_group           32778 non-null  object 
 9   gender              32778 non-null  object 
 10  annual_income_lakh  32778 non-null  float64
 11  payment_mode        32778 non-null  object 
 12  kyc_status          32778 non-null  object 
dtypes: float64(1), int64(2), object(10)
memory usage: 3.3+ MB


In [100]:
df2.describe()

,amfi_code,amount_inr,annual_income_lakh
count,32778.000000,32778.000000,32778.000000
mean,120264.617518,107437.318628,26.181219
std,14370.205345,150415.905084,20.805425
min,100016.000000,400.000000,3.000000
25%,118632.000000,3153.000000,10.600000
50%,119551.000000,17782.500000,19.700000
75%,120843.000000,189324.250000,37.400000
max,149324.000000,597498.000000,99.700000


In [102]:
df2['transaction_date'] = pd.to_datetime(df2['transaction_date'])

In [ ]:
# check transaction type
df2['transaction_type'].value_counts()
# transaction_type values already standardized
# No cleaning required

transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

In [107]:
df2['kyc_status'].value_counts()

kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [108]:
df2['transaction_date'].value_counts()

transaction_date
2025-04-28    88
2025-04-25    87
2024-03-13    86
2024-06-12    86
2024-10-16    85
              ..
2024-09-22    46
2024-07-23    46
2024-06-11    45
2025-04-09    43
2024-11-19    43
Name: count, Length: 516, dtype: int64

In [109]:
# invalid amount
invalid_amount = df2[df2['amount_inr']<=0]
print("Invalid amounts:", len(invalid_amount))

Invalid amounts: 0


In [110]:
df2.isnull().sum()

investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

In [111]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

In [114]:
duplicates = df2.duplicated().sum()
print(duplicates)

0


In [115]:
df2.to_csv(
    "../data/processed/08_investor_transactions_clean.csv",
    index=False
)

In [116]:
clean_df2 = pd.read_csv(
    "../data/processed/08_investor_transactions_clean.csv",
    parse_dates=["transaction_date"]
)

clean_df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32778 entries, 0 to 32777
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   investor_id         32778 non-null  object        
 1   transaction_date    32778 non-null  datetime64[ns]
 2   amfi_code           32778 non-null  int64         
 3   transaction_type    32778 non-null  object        
 4   amount_inr          32778 non-null  int64         
 5   state               32778 non-null  object        
 6   city                32778 non-null  object        
 7   city_tier           32778 non-null  object        
 8   age_group           32778 non-null  object        
 9   gender              32778 non-null  object        
 10  annual_income_lakh  32778 non-null  float64       
 11  payment_mode        32778 non-null  object        
 12  kyc_status          32778 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), ob

## scheme performance ##

In [118]:
df3 = pd.read_csv("../data/raw/07_scheme_performance.csv")
df3.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [119]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   amfi_code           40 non-null     int64  
 1   scheme_name         40 non-null     object 
 2   fund_house          40 non-null     object 
 3   category            40 non-null     object 
 4   plan                40 non-null     object 
 5   return_1yr_pct      40 non-null     float64
 6   return_3yr_pct      40 non-null     float64
 7   return_5yr_pct      40 non-null     float64
 8   benchmark_3yr_pct   40 non-null     float64
 9   alpha               40 non-null     float64
 10  beta                40 non-null     float64
 11  sharpe_ratio        40 non-null     float64
 12  sortino_ratio       40 non-null     float64
 13  std_dev_ann_pct     40 non-null     float64
 14  max_drawdown_pct    40 non-null     float64
 15  aum_crore           40 non-null     int64  
 16  expense_ra

In [120]:
df3['expense_ratio_pct'].describe()

count    40.000000
mean      1.237000
std       0.386584
min       0.550000
25%       0.787500
50%       1.425000
75%       1.540000
max       1.640000
Name: expense_ratio_pct, dtype: float64

In [126]:
#invalid expense ratio

invalid_expense = df3[
    (df3["expense_ratio_pct"] < 0.1) |
    (df3["expense_ratio_pct"] > 2.5)
]

print("Invalid expense ratios:", len(invalid_expense))

Invalid expense ratios: 0


In [127]:
df3["morningstar_rating"].value_counts()

morningstar_rating
5    17
4    16
3     7
Name: count, dtype: int64

In [129]:
df3['risk_grade'].value_counts()

risk_grade
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4
Name: count, dtype: int64

In [130]:
df3.to_csv(
    "../data/processed/07_schema_performance_clean.csv",
    index=False
)